In [0]:
# ============================================
# Stage 4.4: Load DimProduct (Snowflake → Star)
# ============================================

from pyspark.sql import functions as F

base = "abfss://raw@stadventureworksdw.dfs.core.windows.net/AdventureWorks2019/"

df_product      = spark.read.parquet(base + "Product/")
df_subcategory  = spark.read.parquet(base + "ProductSubcategory/")
df_category     = spark.read.parquet(base + "ProductCategory/")

print(f"Product: {df_product.count()} rows")
print(f"ProductSubcategory: {df_subcategory.count()} rows")
print(f"ProductCategory: {df_category.count()} rows")

Product: 504 rows
ProductSubcategory: 37 rows
ProductCategory: 4 rows


In [0]:
# 雪花转星型：三表 JOIN
dim_product = (df_product.alias("p")
    .join(df_subcategory.alias("ps"), 
          F.col("p.ProductSubcategoryID") == F.col("ps.ProductSubcategoryID"), 
          how="left")
    .join(df_category.alias("pc"), 
          F.col("ps.ProductCategoryID") == F.col("pc.ProductCategoryID"), 
          how="left")
    .select(
        F.col("p.ProductID").cast("int").alias("ProductID"),
        F.col("p.Name").alias("ProductName"),
        F.col("p.ProductNumber").alias("ProductNumber"),
        F.col("ps.Name").alias("SubcategoryName"),
        F.col("pc.Name").alias("CategoryName"),
        F.col("p.ListPrice").cast("decimal(19,4)").alias("ListPrice"),
        F.col("p.StandardCost").cast("decimal(19,4)").alias("StandardCost"),
        F.col("p.Color").alias("Color"),
        F.col("p.Size").alias("Size"),
        F.current_timestamp().alias("valid_from"),
        F.to_date(F.lit("9999-12-31")).alias("valid_to"),
        F.lit(1).cast("int").alias("is_current")
    )
    .withColumn("ProductKey", F.monotonically_increasing_id() + 1)
    .select("ProductKey", "ProductID", "ProductName", "ProductNumber",
            "SubcategoryName", "CategoryName", "ListPrice", "StandardCost",
            "Color", "Size", "valid_from", "valid_to", "is_current")
)

print(f"Total rows: {dim_product.count()}")
display(dim_product.limit(10))

Total rows: 504


ProductKey,ProductID,ProductName,ProductNumber,SubcategoryName,CategoryName,ListPrice,StandardCost,Color,Size,valid_from,valid_to,is_current
1,1,Adjustable Race,AR-5381,null,null,0.0000,0.0000,null,null,2026-09-27T07:04:37.559Z,9999-12-31,1
2,2,Bearing Ball,BA-8327,null,null,0.0000,0.0000,null,null,2026-09-27T07:04:37.559Z,9999-12-31,1
3,3,BB Ball Bearing,BE-2349,null,null,0.0000,0.0000,null,null,2026-09-27T07:04:37.559Z,9999-12-31,1
4,4,Headset Ball Bearings,BE-2908,null,null,0.0000,0.0000,null,null,2026-09-27T07:04:37.559Z,9999-12-31,1
5,316,Blade,BL-2036,null,null,0.0000,0.0000,null,null,2026-09-27T07:04:37.559Z,9999-12-31,1
6,317,LL Crankarm,CA-5965,null,null,0.0000,0.0000,Black,null,2026-09-27T07:04:37.559Z,9999-12-31,1
7,318,ML Crankarm,CA-6738,null,null,0.0000,0.0000,Black,null,2026-09-27T07:04:37.559Z,9999-12-31,1
8,319,HL Crankarm,CA-7457,null,null,0.0000,0.0000,Black,null,2026-09-27T07:04:37.559Z,9999-12-31,1
9,320,Chainring Bolts,CB-2903,null,null,0.0000,0.0000,Silver,null,2026-09-27T07:04:37.559Z,9999-12-31,1
10,321,Chainring Nut,CN-6137,null,null,0.0000,0.0000,Silver,null,2026-09-27T07:04:37.559Z,9999-12-31,1


In [0]:
# 验证类别分布
dim_product.groupBy("CategoryName") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

+------------+-----+
|CategoryName|count|
+------------+-----+
|        NULL|  209|
|  Components|  134|
|       Bikes|   97|
|    Clothing|   35|
| Accessories|   29|
+------------+-----+



In [0]:
# 写入 Curated Zone
curated_path = "abfss://curated@stadventureworksdw.dfs.core.windows.net/AdventureWorksDW/DimProduct/"

dim_product.write \
    .format("delta") \
    .mode("overwrite") \
    .save(curated_path)

print(f"✅ DimProduct written to {curated_path}")

✅ DimProduct written to abfss://curated@stadventureworksdw.dfs.core.windows.net/AdventureWorksDW/DimProduct/


In [0]:
# 验证
df_check = spark.read.format("delta").load(curated_path)
print(f"Total rows: {df_check.count()}")
df_check.show(5)

Total rows: 504
+----------+---------+--------------------+-------------+---------------+------------+---------+------------+-----+----+--------------------+----------+----------+
|ProductKey|ProductID|         ProductName|ProductNumber|SubcategoryName|CategoryName|ListPrice|StandardCost|Color|Size|          valid_from|  valid_to|is_current|
+----------+---------+--------------------+-------------+---------------+------------+---------+------------+-----+----+--------------------+----------+----------+
|         1|        1|     Adjustable Race|      AR-5381|           NULL|        NULL|   0.0000|      0.0000| NULL|NULL|2026-09-27 07:05:...|9999-12-31|         1|
|         2|        2|        Bearing Ball|      BA-8327|           NULL|        NULL|   0.0000|      0.0000| NULL|NULL|2026-09-27 07:05:...|9999-12-31|         1|
|         3|        3|     BB Ball Bearing|      BE-2349|           NULL|        NULL|   0.0000|      0.0000| NULL|NULL|2026-09-27 07:05:...|9999-12-31|         1|
